### Train Data - Feature Engineering

In [ ]:
# Import the Libraries & Load the Cleaned Dataset
import pandas as pd
import os

RAW_DATA_PATH = os.path.join(r"C:\Users\nikun\Documents\Demand Forecasting\data", "raw")
PROCESSED_DATA_PATH = os.path.join(r"C:\Users\nikun\Documents\Demand Forecasting\data", "processed")

train_df = pd.read_csv(os.path.join(PROCESSED_DATA_PATH, "train_eda_processed.csv"))
test_df = pd.read_csv(os.path.join(RAW_DATA_PATH, "test.csv"))
store_df = pd.read_csv(os.path.join(PROCESSED_DATA_PATH, "store_eda_processed.csv"))

C:\Users\nikun\AppData\Local\Temp\ipykernel_6456\4229561763.py:8: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(os.path.join(PROCESSED_DATA_PATH, "train_eda_processed.csv"))


In [195]:
# Display the Structure of the Train DataFrame
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 13 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   Store               1017209 non-null  int64  
 1   DayOfWeek           1017209 non-null  int64  
 2   Date                1017209 non-null  object 
 3   Sales               1017209 non-null  int64  
 4   Customers           1017209 non-null  int64  
 5   Open                1017209 non-null  int64  
 6   Promo               1017209 non-null  int64  
 7   StateHoliday        1017209 non-null  object 
 8   SchoolHoliday       1017209 non-null  int64  
 9   Year                1017209 non-null  int64  
 10  Month               1017209 non-null  int64  
 11  Day                 1017209 non-null  int64  
 12  Sales_per_Customer  844340 non-null   float64
dtypes: float64(1), int64(10), object(2)
memory usage: 100.9+ MB


In [196]:
# Extract Custom Features from Date
train_df['Date'] = pd.to_datetime(train_df['Date'])
train_df['WeekOfYear'] = train_df['Date'].dt.isocalendar().week.astype(int)
train_df['Quarter'] = train_df['Date'].dt.quarter.astype(int)
train_df['IsWeekend'] = train_df['Date'].dt.dayofweek.isin([5, 6]).astype(int)

# Check DataFrame Structure post adding Custom Features
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 16 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   Store               1017209 non-null  int64         
 1   DayOfWeek           1017209 non-null  int64         
 2   Date                1017209 non-null  datetime64[ns]
 3   Sales               1017209 non-null  int64         
 4   Customers           1017209 non-null  int64         
 5   Open                1017209 non-null  int64         
 6   Promo               1017209 non-null  int64         
 7   StateHoliday        1017209 non-null  object        
 8   SchoolHoliday       1017209 non-null  int64         
 9   Year                1017209 non-null  int64         
 10  Month               1017209 non-null  int64         
 11  Day                 1017209 non-null  int64         
 12  Sales_per_Customer  844340 non-null   float64       
 13  WeekOfYear  

In [197]:
# Drop Unnecessary Columns - Date & Sales_per_Customer
train_df = train_df.drop(columns=["Date", "Sales_per_Customer"])

# Apply One-Hot Encoding to Categorical Variables - DayOfWeek, Open, Promo, SchoolHoliday
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder()
categorical_cols = ["DayOfWeek", "Open", "Promo", "SchoolHoliday"]
encoded_data = encoder.fit_transform(train_df[["DayOfWeek", "Open", "Promo", "SchoolHoliday"]]).toarray()

# Create a DataFrame with the Encoded Columns
encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(categorical_cols))
encoded_df.index = train_df.index

# Convert Encoded Columns to Integer Type
encoded_df = encoded_df.astype(int)

# Concatenate the Encoded DataFrame with the Original DataFrame
train_df = pd.concat([train_df.drop(columns=categorical_cols), encoded_df], axis=1)

# Handle Datatyper Conversion for StateHoliday
train_df["StateHoliday"] = train_df["StateHoliday"].astype(str)

# Apply One-Hot Encoding to StateHoliday
encoder_state_holiday = OneHotEncoder()
state_holiday_encoded = encoder_state_holiday.fit_transform(train_df[["StateHoliday"]]).toarray()

# Create a DataFrame with the Encoded StateHoliday Columns
column_names = [f"StateHoliday_{cat}" for cat in encoder_state_holiday.categories_[0]]
state_holiday_df = pd.DataFrame(state_holiday_encoded, columns=column_names)
state_holiday_df.index = train_df.index

# Convert Encoded Columns to Integer Type
state_holiday_df = state_holiday_df.astype(int)

# Concatenate the Encoded StateHoliday DataFrame with the Original DataFrame
train_df = pd.concat([train_df.drop(columns=["StateHoliday"]), state_holiday_df], axis=1)
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 26 columns):
 #   Column           Non-Null Count    Dtype
---  ------           --------------    -----
 0   Store            1017209 non-null  int64
 1   Sales            1017209 non-null  int64
 2   Customers        1017209 non-null  int64
 3   Year             1017209 non-null  int64
 4   Month            1017209 non-null  int64
 5   Day              1017209 non-null  int64
 6   WeekOfYear       1017209 non-null  int64
 7   Quarter          1017209 non-null  int64
 8   IsWeekend        1017209 non-null  int64
 9   DayOfWeek_1      1017209 non-null  int64
 10  DayOfWeek_2      1017209 non-null  int64
 11  DayOfWeek_3      1017209 non-null  int64
 12  DayOfWeek_4      1017209 non-null  int64
 13  DayOfWeek_5      1017209 non-null  int64
 14  DayOfWeek_6      1017209 non-null  int64
 15  DayOfWeek_7      1017209 non-null  int64
 16  Open_0           1017209 non-null  int64
 17  Open_1  

In [198]:
# Save the Processed Train DataFrame
train_df.to_csv(os.path.join(PROCESSED_DATA_PATH, "train_feature_engineered.csv"), index=False)

### Store Data - Feature Engineering

In [199]:
# Display the Structure of the Store DataFrame
store_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Store                      1115 non-null   int64  
 1   StoreType                  1115 non-null   object 
 2   Assortment                 1115 non-null   object 
 3   CompetitionDistance        1115 non-null   float64
 4   CompetitionOpenSinceMonth  1115 non-null   float64
 5   CompetitionOpenSinceYear   1115 non-null   float64
 6   Promo2                     1115 non-null   int64  
 7   Promo2SinceWeek            1115 non-null   float64
 8   Promo2SinceYear            1115 non-null   float64
 9   PromoInterval              1115 non-null   object 
dtypes: float64(5), int64(2), object(3)
memory usage: 87.2+ KB


In [200]:
# Apply OneHotEncoder to Categorical Columns - StoreType, Assortment & Promo2
encoder_storetype_assortment_promo2 = OneHotEncoder()
categorical_cols_storetype_assortment_promo2 = ["StoreType", "Assortment", "Promo2"]
encoded_storetype_assortment_promo2 = encoder_storetype_assortment_promo2.fit_transform(store_df[['StoreType', 'Assortment', 'Promo2']]).toarray()

# Create a DataFrame with the Encoded Columns
encoded_storetype_assortment_promo2_df = pd.DataFrame(encoded_storetype_assortment_promo2, columns=encoder_storetype_assortment_promo2.get_feature_names_out(categorical_cols_storetype_assortment_promo2))
encoded_storetype_assortment_promo2_df.index = store_df.index

# Convert Encoded Columns to Integer Type
encoded_storetype_assortment_promo2_df = encoded_storetype_assortment_promo2_df.astype(int)

# Concatenate the Encoded DataFrame with the Original DataFrame & Drop Categorical Features
store_df = pd.concat([store_df.drop(columns=categorical_cols_storetype_assortment_promo2), encoded_storetype_assortment_promo2_df], axis=1)

# Extract all possible months that appear in PromoInterval
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sept','Oct','Nov','Dec']

# Create binary columns for each month
for month in months:
    store_df[f'Promo_{month}'] = store_df['PromoInterval'].apply(lambda x: 1 if month in str(x) else 0)

# Drop the original column - PromoInterval
store_df = store_df.drop('PromoInterval', axis=1)

# Handle the Columns - CompetitionOpenSinceMonth, CompetitionOpenSinceYear, Promo2SinceWeek, Promo2SinceYear
store_df['CompetitionOpenSinceMonth'] = store_df['CompetitionOpenSinceMonth'].fillna(0).astype(int)
store_df['CompetitionOpenSinceYear'] = store_df['CompetitionOpenSinceYear'].fillna(0).astype(int)
store_df['Promo2SinceWeek'] = store_df['Promo2SinceWeek'].fillna(0).astype(int)
store_df['Promo2SinceYear'] = store_df['Promo2SinceYear'].fillna(0).astype(int)
store_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Store                      1115 non-null   int64  
 1   CompetitionDistance        1115 non-null   float64
 2   CompetitionOpenSinceMonth  1115 non-null   int64  
 3   CompetitionOpenSinceYear   1115 non-null   int64  
 4   Promo2SinceWeek            1115 non-null   int64  
 5   Promo2SinceYear            1115 non-null   int64  
 6   StoreType_a                1115 non-null   int64  
 7   StoreType_b                1115 non-null   int64  
 8   StoreType_c                1115 non-null   int64  
 9   StoreType_d                1115 non-null   int64  
 10  Assortment_a               1115 non-null   int64  
 11  Assortment_b               1115 non-null   int64  
 12  Assortment_c               1115 non-null   int64  
 13  Promo2_0                   1115 non-null   int64

In [201]:
# --- Handle CompetitionDistance ---

import numpy as np

# Log transform (since NaNs are already handled in EDA)
store_df['CompetitionDistance_log'] = np.log1p(store_df['CompetitionDistance'])

# Binning (optional, for categorical representation)
store_df['CompetitionDistance_bin'] = pd.qcut(
    store_df['CompetitionDistance'], 
    q=4, 
    labels=['Very_Close','Close','Far','Very_Far']
)

In [202]:
# Introduce New Features for Cyclical Encoding & Age

# --- Cyclical encoding ---
store_df['CompMonth_sin'] = np.sin(2 * np.pi * store_df['CompetitionOpenSinceMonth']/12)
store_df['CompMonth_cos'] = np.cos(2 * np.pi * store_df['CompetitionOpenSinceMonth']/12)

store_df['PromoWeek_sin'] = np.sin(2 * np.pi * store_df['Promo2SinceWeek']/52)
store_df['PromoWeek_cos'] = np.cos(2 * np.pi * store_df['Promo2SinceWeek']/52)

# --- Age features ---
current_year = 2015  # adjust based on dataset reference year
current_month = 8    # adjust based on dataset reference month
current_week = 31    # adjust based on dataset reference week

store_df['CompetitionAgeMonths'] = (current_year - store_df['CompetitionOpenSinceYear'])*12 + \
                                   (current_month - store_df['CompetitionOpenSinceMonth'])

store_df['Promo2AgeWeeks'] = (current_year - store_df['Promo2SinceYear'])*52 + \
                             (current_week - store_df['Promo2SinceWeek'])

In [203]:
# Display the Structure of the Store DataFrame
store_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 35 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   Store                      1115 non-null   int64   
 1   CompetitionDistance        1115 non-null   float64 
 2   CompetitionOpenSinceMonth  1115 non-null   int64   
 3   CompetitionOpenSinceYear   1115 non-null   int64   
 4   Promo2SinceWeek            1115 non-null   int64   
 5   Promo2SinceYear            1115 non-null   int64   
 6   StoreType_a                1115 non-null   int64   
 7   StoreType_b                1115 non-null   int64   
 8   StoreType_c                1115 non-null   int64   
 9   StoreType_d                1115 non-null   int64   
 10  Assortment_a               1115 non-null   int64   
 11  Assortment_b               1115 non-null   int64   
 12  Assortment_c               1115 non-null   int64   
 13  Promo2_0                   1115 n

In [204]:
# --- Final cleanup for store_df ---

# Drop raw columns that are already transformed
store_df = store_df.drop([
    'CompetitionOpenSinceMonth',
    'CompetitionOpenSinceYear',
    'Promo2SinceWeek',
    'Promo2SinceYear'
], axis=1)

# Optional: One-hot encode the bins if needed for your model
store_df = pd.get_dummies(store_df, columns=['CompetitionDistance_bin'], drop_first=True)

# Convert Datatypes to Integer
store_df = store_df.astype({ 
    'CompetitionDistance_bin_Close': 'int',
    'CompetitionDistance_bin_Far': 'int',
    'CompetitionDistance_bin_Very_Far': 'int'
})
store_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 33 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Store                             1115 non-null   int64  
 1   CompetitionDistance               1115 non-null   float64
 2   StoreType_a                       1115 non-null   int64  
 3   StoreType_b                       1115 non-null   int64  
 4   StoreType_c                       1115 non-null   int64  
 5   StoreType_d                       1115 non-null   int64  
 6   Assortment_a                      1115 non-null   int64  
 7   Assortment_b                      1115 non-null   int64  
 8   Assortment_c                      1115 non-null   int64  
 9   Promo2_0                          1115 non-null   int64  
 10  Promo2_1                          1115 non-null   int64  
 11  Promo_Jan                         1115 non-null   int64  
 12  Promo_

In [205]:
# Save the Processed Store DataFrame
store_df.to_csv(os.path.join(PROCESSED_DATA_PATH, "store_feature_engineered.csv"), index=False)

### Test Data - Feature Engineering

In [206]:
# Display Structure of Test DataFrame
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41088 entries, 0 to 41087
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             41088 non-null  int64  
 1   Store          41088 non-null  int64  
 2   DayOfWeek      41088 non-null  int64  
 3   Date           41088 non-null  object 
 4   Open           41077 non-null  float64
 5   Promo          41088 non-null  int64  
 6   StateHoliday   41088 non-null  object 
 7   SchoolHoliday  41088 non-null  int64  
dtypes: float64(1), int64(5), object(2)
memory usage: 2.5+ MB


In [207]:
# Handle Missing Values
test_df["Open"].fillna(1, inplace=True)   # Closed stores treated as open unless specified

# Extract Year, Month & Day from Date
test_df['Date'] = pd.to_datetime(test_df['Date'])
test_df["Year"] = test_df["Date"].dt.year.astype(int)
test_df["Month"] = test_df["Date"].dt.month.astype(int)
test_df["Day"] = test_df["Date"].dt.day.astype(int)
test_df['WeekOfYear'] = test_df['Date'].dt.isocalendar().week.astype(int)
test_df['Quarter'] = test_df['Date'].dt.quarter.astype(int)
test_df['IsWeekend'] = test_df['Date'].dt.dayofweek.isin([5, 6]).astype(int)

# Convert Datatype for Feature - Open
test_df["Open"] = test_df["Open"].astype(int)

# Apply One Hot Encoding to Features - Open, Promo, SchoolHoliday, DayOfWeek
encoder_test_open_promo_schoolholiday_dayofweek = OneHotEncoder()
categorical_cols_test_open_promo_schoolholiday_dayofweek = ["Open", "Promo", "SchoolHoliday", "DayOfWeek"]
encoded_test_open_promo_schoolholiday_dayofweek = encoder_test_open_promo_schoolholiday_dayofweek.fit_transform(test_df[categorical_cols_test_open_promo_schoolholiday_dayofweek]).toarray()

# Create DataFrame for Encoded Features
encoded_df_test_open_promo_schoolholiday_dayofweek = pd.DataFrame(encoded_test_open_promo_schoolholiday_dayofweek, columns=encoder_test_open_promo_schoolholiday_dayofweek.get_feature_names_out(categorical_cols_test_open_promo_schoolholiday_dayofweek))

# Convert the DataFrame to the appropriate data types
encoded_df_test_open_promo_schoolholiday_dayofweek = encoded_df_test_open_promo_schoolholiday_dayofweek.astype(int)

# Concatenate with Original DataFrame
test_df = pd.concat([test_df, encoded_df_test_open_promo_schoolholiday_dayofweek], axis=1)

# Drop Original Categorical Columns
test_df.drop(categorical_cols_test_open_promo_schoolholiday_dayofweek, axis=1, inplace=True)
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41088 entries, 0 to 41087
Data columns (total 23 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Id               41088 non-null  int64         
 1   Store            41088 non-null  int64         
 2   Date             41088 non-null  datetime64[ns]
 3   StateHoliday     41088 non-null  object        
 4   Year             41088 non-null  int64         
 5   Month            41088 non-null  int64         
 6   Day              41088 non-null  int64         
 7   WeekOfYear       41088 non-null  int64         
 8   Quarter          41088 non-null  int64         
 9   IsWeekend        41088 non-null  int64         
 10  Open_0           41088 non-null  int64         
 11  Open_1           41088 non-null  int64         
 12  Promo_0          41088 non-null  int64         
 13  Promo_1          41088 non-null  int64         
 14  SchoolHoliday_0  41088 non-null  int64

C:\Users\nikun\AppData\Local\Temp\ipykernel_6456\4287785374.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_df["Open"].fillna(1, inplace=True)   # Closed stores treated as open unless specified


In [208]:
# Ensure test has the same StateHoliday dummies as train
for col in ["StateHoliday_b", "StateHoliday_c"]:
    if col not in test_df.columns:
        test_df[col] = 0

# Apply One Hot Encoding for StateHoliday
encoder_test_stateholiday = OneHotEncoder()
categorical_cols_test_stateholiday = ["StateHoliday"]
encoded_test_stateholiday = encoder_test_stateholiday.fit_transform(test_df[categorical_cols_test_stateholiday]).toarray()

# Create DataFrame for Encoded Features
encoded_df_test_stateholiday = pd.DataFrame(encoded_test_stateholiday, columns=encoder_test_stateholiday.get_feature_names_out(categorical_cols_test_stateholiday))

# Convert the DataFrame to the appropriate data types
encoded_df_test_stateholiday = encoded_df_test_stateholiday.astype(int)

# Concatenate with Original DataFrame
test_df = pd.concat([test_df, encoded_df_test_stateholiday], axis=1)

# Drop Original Categorical Columns
test_df.drop(categorical_cols_test_stateholiday, axis=1, inplace=True)
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41088 entries, 0 to 41087
Data columns (total 26 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Id               41088 non-null  int64         
 1   Store            41088 non-null  int64         
 2   Date             41088 non-null  datetime64[ns]
 3   Year             41088 non-null  int64         
 4   Month            41088 non-null  int64         
 5   Day              41088 non-null  int64         
 6   WeekOfYear       41088 non-null  int64         
 7   Quarter          41088 non-null  int64         
 8   IsWeekend        41088 non-null  int64         
 9   Open_0           41088 non-null  int64         
 10  Open_1           41088 non-null  int64         
 11  Promo_0          41088 non-null  int64         
 12  Promo_1          41088 non-null  int64         
 13  SchoolHoliday_0  41088 non-null  int64         
 14  SchoolHoliday_1  41088 non-null  int64

In [209]:
# Drop the Date Column
test_df.drop("Date", axis=1, inplace=True)

In [210]:
# Save the Processed Test DataFrame
test_df.to_csv(os.path.join(PROCESSED_DATA_PATH, "test_feature_engineered.csv"), index=False)

### Comparision of Train vs Test Data

In [211]:
# Make DataFrame comparison - Train vs Test
train_cols = set(train_df.columns)
test_cols = set(test_df.columns)

comparison = pd.DataFrame({
    "In_Train": [col in train_cols for col in sorted(train_cols | test_cols)],
    "In_Test": [col in test_cols for col in sorted(train_cols | test_cols)]
}, index=sorted(train_cols | test_cols))

print("Comparison of Train and Test DataFrames")
print(comparison)

Comparison of Train and Test DataFrames
                 In_Train  In_Test
Customers            True    False
Day                  True     True
DayOfWeek_1          True     True
DayOfWeek_2          True     True
DayOfWeek_3          True     True
DayOfWeek_4          True     True
DayOfWeek_5          True     True
DayOfWeek_6          True     True
DayOfWeek_7          True     True
Id                  False     True
IsWeekend            True     True
Month                True     True
Open_0               True     True
Open_1               True     True
Promo_0              True     True
Promo_1              True     True
Quarter              True     True
Sales                True    False
SchoolHoliday_0      True     True
SchoolHoliday_1      True     True
StateHoliday_0       True     True
StateHoliday_a       True     True
StateHoliday_b       True     True
StateHoliday_c       True     True
Store                True     True
WeekOfYear           True     True
Year           

In [212]:
# Handle Mismatches between Train and Test

# Make copies to be safe
train_ready = train_df.copy()
test_ready = test_df.copy()

# Drop target (Sales) and Customers from train
X_train = train_ready.drop(["Sales", "Customers"], axis=1)
y_train = train_ready["Sales"]   # keep Sales separately as target

# Drop Id from test, but keep a copy for submission
test_ids = test_ready["Id"]
X_test = test_ready.drop(["Id"], axis=1)

print("Train features shape:", X_train.shape)
print("Train target shape:", y_train.shape)
print("Test features shape:", X_test.shape)

Train features shape: (1017209, 24)
Train target shape: (1017209,)
Test features shape: (41088, 24)


In [213]:
# Verify if the Column Mismatches have been handled for Train & Test

# Verify train and test column alignment
train_cols = set(X_train.columns)
test_cols = set(X_test.columns)

print("Columns in Train but not in Test:", train_cols - test_cols)
print("Columns in Test but not in Train:", test_cols - train_cols)

# Reorder test columns to match train
X_test = X_test[X_train.columns]

# Double-check again
assert list(X_train.columns) == list(X_test.columns), "Mismatch still exists!"
print("✅ Train and Test feature columns now match perfectly!")

Columns in Train but not in Test: set()
Columns in Test but not in Train: set()
✅ Train and Test feature columns now match perfectly!


### Merging of Train & Test with Store

In [214]:
# Load Processed Data
train = pd.read_csv(os.path.join(PROCESSED_DATA_PATH, "train_feature_engineered.csv"))
test = pd.read_csv(os.path.join(PROCESSED_DATA_PATH, "test_feature_engineered.csv"))
store = pd.read_csv(os.path.join(PROCESSED_DATA_PATH, "store_feature_engineered.csv"))

# Merge Train & Test with Store
train_merged = train.merge(store, on="Store", how="left")
test_merged = test.merge(store, on="Store", how="left")

# --------------------
# Align columns between train and test
# --------------------
# Save target columns
target_cols = ["Sales", "Customers"]  # only in train

# Drop targets temporarily for alignment
train_X = train_merged.drop(columns=target_cols, errors="ignore")
test_X = test_merged.copy()

# Make sure test has all train columns (except targets)
for col in train_X.columns:
    if col not in test_X.columns:
        test_X[col] = 0  # or np.nan → depends on how you want to handle

# Make sure train has all test columns
for col in test_X.columns:
    if col not in train_X.columns:
        train_X[col] = 0

# Reorder columns identically
train_X = train_X.reindex(sorted(train_X.columns), axis=1)
test_X = test_X.reindex(sorted(test_X.columns), axis=1)

# Add target columns back to train
for col in target_cols:
    if col in train_merged.columns:
        train_X[col] = train_merged[col]

# --------------------
# Save final datasets
# --------------------
train_X.to_csv(os.path.join(PROCESSED_DATA_PATH, "train_store_merged.csv"), index=False)
test_X.to_csv(os.path.join(PROCESSED_DATA_PATH, "test_store_merged.csv"), index=False)

print("✅ Final datasets saved!")
print("Train shape:", train_X.shape)
print("Test shape:", test_X.shape)

✅ Final datasets saved!
Train shape: (1017209, 59)
Test shape: (41088, 57)


In [215]:
train_cols = set(train_X.drop(columns=["Sales","Customers"]).columns)
test_cols = set(test_X.columns)

print("Columns only in Train:", train_cols - test_cols)
print("Columns only in Test:", test_cols - train_cols)

Columns only in Train: set()
Columns only in Test: set()
